<a href="https://colab.research.google.com/github/shravan1808/ML_SERIES/blob/Main/10_Cross_Validation_Overfitting_Diagnostics_Pipeline/notebook/Project_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import numpy as np

In [20]:
policyholder_data = {
    'Policy_ID': [f'POL_{700+i}' for i in range(50)],
    'Driver_Age': [22, 45, 34, 60, 28, 50, 19, 41, 33, 55,
                   24, 48, 31, 62, 26, 52, 20, 43, 36, 58,
                   23, 46, 32, 61, 27, 51, 21, 42, 35, 56,
                   25, 49, 30, 63, 29, 53, 22, 44, 37, 59,
                   24, 47, 33, 64, 28, 54, 20, 40, 38, 57],
    'Credit_Score': [610, 750, 680, 810, 640, 780, 580, 720, 690, 790,
                     620, 760, 670, 820, 630, 770, 590, 730, 700, 800,
                     615, 755, 675, 815, 635, 775, 585, 725, 695, 795,
                     625, 765, 665, 825, 645, 785, 600, 735, 705, 805,
                     618, 758, 678, 818, 638, 778, 592, 722, 698, 788]
}

In [21]:
vehicle_data = {
    'Policy_ID': [f'POL_{700+i}' for i in range(50)],
    'Vehicle_Age_Yrs': [12, 3, 7, 1, 9, 2, 14, 4, 6, 2,
                        11, 2, 8, 1, 10, 3, 13, 5, 6, 1,
                        12, 3, 7, 1, 9, 2, 14, 4, 5, 2,
                        10, 2, 8, 1, 9, 3, 13, 4, 6, 1,
                        11, 3, 7, 1, 10, 2, 14, 5, 6, 2],
    'Annual_Mileage_KM': [22000, 11000, 15000, 8000, 18000, 9500, 25000, 12000, 14000, 9000,
                          21000, 10500, 16000, 7500, 19000, 9800, 24000, 12500, 14500, 8500,
                          22500, 11200, 15200, 8200, 18200, 9600, 25500, 12200, 13800, 9100,
                          20500, 10800, 16200, 7800, 18800, 10000, 24500, 12800, 14200, 8800,
                          21800, 11100, 15800, 8100, 18500, 9700, 25200, 12100, 14100, 9200]
}

In [22]:
claim_data = {
    'Policy_ID': [f'POL_{700+i}' for i in range(50)],
    'Made_Claim': [1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
                   1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
                   1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
                   1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
                   1, 0, 0, 0, 1, 0, 1, 0, 0, 0] # 15 Claims (30%), 35 No Claims (70%)
}

In [23]:
df_policy = pd.DataFrame(policyholder_data)
df_vehicle = pd.DataFrame(vehicle_data)
df_claim = pd.DataFrame(claim_data)

In [24]:
final_df = (
    df_policy
    .merge(df_vehicle,on ='Policy_ID',how='inner')
    .merge(df_claim,on ='Policy_ID',how='inner')
)

In [25]:
print(final_df.head().to_string())

  Policy_ID  Driver_Age  Credit_Score  Vehicle_Age_Yrs  Annual_Mileage_KM  Made_Claim
0   POL_700          22           610               12              22000           1
1   POL_701          45           750                3              11000           0
2   POL_702          34           680                7              15000           0
3   POL_703          60           810                1               8000           0
4   POL_704          28           640                9              18000           1


In [26]:
X= final_df.drop(['Policy_ID','Made_Claim'],axis=1)
y= final_df['Made_Claim']

In [27]:
from sklearn.model_selection import KFold

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

kfold_results = []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X),
    start=1
):
    y_val = y.iloc[val_idx]

    validation_records = len(y_val)
    claims = y_val.sum()
    claim_percentage = y_val.mean() * 100

    kfold_results.append({
        'fold': fold,
        'records': validation_records,
        'claims': claims,
        'percentage': claim_percentage
    })

In [28]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

stratified_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X, y),
    start=1
):
    y_val = y.iloc[val_idx]

    validation_records = len(y_val)
    claims = y_val.sum()
    claim_percentage = y_val.mean() * 100

    stratified_results.append({
        'fold': fold,
        'records': validation_records,
        'claims': claims,
        'percentage': claim_percentage
    })

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

accuracy_scores = []

for train_idx, val_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_val)

    # Evaluate
    accuracy = accuracy_score(y_val, y_pred)

    accuracy_scores.append(accuracy)

mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)

In [32]:
print("=" * 10, "CROSS-VALIDATION & OVERFITTING DIAGNOSTICS", "=" * 10)

print()

print(f"{'Master Insurance Dataset Records':<35} : {len(final_df)}")

print(
    f"{'Features (X)':<35} : "
    f"{', '.join(X.columns)}"
)

positive_rate = y.mean() * 100

print(
    f"{'Target (y)':<35} : "
    f"{y.name} ({positive_rate:.1f}% Positive Rate)"
)

print("\nCross-Validation Comparison:")

kfold_percentages = [
    result['percentage']
    for result in kfold_results
]

stratified_percentages = [
    result['percentage']
    for result in stratified_results
]

print(
    f"- {'Standard K-Fold (5 Splits)':<32} : "
    f"High fold variance "
    f"(Validation positive rate: "
    f"{min(kfold_percentages):.0f}% - "
    f"{max(kfold_percentages):.0f}%)"
)

if len(set(stratified_percentages)) == 1:
    stratified_description = (
        f"Zero fold variance "
        f"(Validation positive rate fixed at "
        f"{stratified_percentages[0]:.0f}%)"
    )
else:
    stratified_description = (
        f"Low fold variance "
        f"(Validation positive rate: "
        f"{min(stratified_percentages):.0f}% - "
        f"{max(stratified_percentages):.0f}%)"
    )

print(
    f"- {'Stratified K-Fold (5 Splits)':<32} : "
    f"{stratified_description}"
)

print("\nBaseline Evaluation Results:")

print(
    f"- {'Mean CV Accuracy Score':<32} : "
    f"{mean_accuracy * 100:.1f}% "
    f"(+/- {std_accuracy * 100:.1f}%)"
)

print("\nConclusion:")

print(
    "Stratified K-Fold Cross-Validation ensures robust, "
    "low-variance evaluation on imbalanced risk data by "
    "maintaining consistent label distributions across "
    "all cross-validation folds."
)

========== CROSS-VALIDATION & OVERFITTING DIAGNOSTICS ==========

Master Insurance Dataset Records    : 50
Features (X)                        : Driver_Age, Credit_Score, Vehicle_Age_Yrs, Annual_Mileage_KM
Target (y)                          : Made_Claim (30.0% Positive Rate)

Cross-Validation Comparison:
- Standard K-Fold (5 Splits)       : High fold variance (Validation positive rate: 20% - 50%)
- Stratified K-Fold (5 Splits)     : Zero fold variance (Validation positive rate fixed at 30%)

Baseline Evaluation Results:
- Mean CV Accuracy Score           : 100.0% (+/- 0.0%)

Conclusion:
Stratified K-Fold Cross-Validation ensures robust, low-variance evaluation on imbalanced risk data by maintaining consistent label distributions across all cross-validation folds.
